<a href="https://colab.research.google.com/github/furkankoc426/ai-deployment-learning/blob/main/notebooks/ai_deployment_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [2]:
import time
import statistics

import torch
from torchvision.models import resnet18, ResNet18_Weights


def print_environment():
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)
    else:
        print("No GPU")


def benchmark_model(model, x, device, warmup_iters=10, measure_iters=50):
    model.eval()
    model.to(device)
    x = x.to(device)

    # Warmup
    with torch.no_grad():
        for _ in range(warmup_iters):
            model(x)

    if device.type == "cuda":
        torch.cuda.synchronize()

    latencies_ms = []

    # Real measurement
    with torch.no_grad():
        for _ in range(measure_iters):
            if device.type == "cuda":
                torch.cuda.synchronize()

            start = time.time()
            model(x)

            if device.type == "cuda":
                torch.cuda.synchronize()

            end = time.time()
            latencies_ms.append((end - start) * 1000)

    avg_latency = statistics.mean(latencies_ms)
    p50_latency = statistics.median(latencies_ms)
    p95_latency = sorted(latencies_ms)[int(len(latencies_ms) * 0.95) - 1]
    throughput = 1000.0 / avg_latency * x.shape[0]

    return {
        "batch_size": x.shape[0],
        "avg_latency_ms": avg_latency,
        "p50_latency_ms": p50_latency,
        "p95_latency_ms": p95_latency,
        "throughput_images_per_second": throughput,
    }


print_environment()

model = resnet18(weights=ResNet18_Weights.DEFAULT)

cpu_device = torch.device("cpu")
gpu_device = torch.device("cuda") if torch.cuda.is_available() else None

for batch_size in [1, 8, 16]:
    x = torch.randn(batch_size, 3, 224, 224)

    print("\nCPU benchmark - batch size:", batch_size)
    cpu_result = benchmark_model(model, x, cpu_device)
    print(cpu_result)

    if gpu_device is not None:
        print("\nGPU benchmark - batch size:", batch_size)
        gpu_result = benchmark_model(model, x, gpu_device)
        print(gpu_result)

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 220MB/s]



CPU benchmark - batch size: 1
{'batch_size': 1, 'avg_latency_ms': 59.506378173828125, 'p50_latency_ms': 57.98029899597168, 'p95_latency_ms': 72.20745086669922, 'throughput_images_per_second': 16.80492126539498}

GPU benchmark - batch size: 1
{'batch_size': 1, 'avg_latency_ms': 3.9059925079345703, 'p50_latency_ms': 3.8787126541137695, 'p95_latency_ms': 3.897428512573242, 'throughput_images_per_second': 256.01687611243904}

CPU benchmark - batch size: 8
{'batch_size': 8, 'avg_latency_ms': 443.71097564697266, 'p50_latency_ms': 407.3578119277954, 'p95_latency_ms': 599.0445613861084, 'throughput_images_per_second': 18.029754590440866}

GPU benchmark - batch size: 8
{'batch_size': 8, 'avg_latency_ms': 10.132150650024414, 'p50_latency_ms': 9.274005889892578, 'p95_latency_ms': 13.133525848388672, 'throughput_images_per_second': 789.5658361515502}

CPU benchmark - batch size: 16
{'batch_size': 16, 'avg_latency_ms': 927.1161508560181, 'p50_latency_ms': 872.697114944458, 'p95_latency_ms': 1232.2

In [6]:
!pip install onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 18.9 MB/s eta 0:00:00


In [7]:
import os
import torch
import onnx

from torchvision.models import resnet18, ResNet18_Weights


print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.eval()

x = torch.randn(1, 3, 224, 224)

onnx_path = "resnet18_static.onnx"

torch.onnx.export(
    model,
    x,
    onnx_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
)

onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

file_size_mb = os.path.getsize(onnx_path) / (1024 * 1024)

print("ONNX export successful:", onnx_path)
print("Input name:", onnx_model.graph.input[0].name)
print("Output name:", onnx_model.graph.output[0].name)
print("Number of nodes:", len(onnx_model.graph.node))
print("File size MB:", round(file_size_mb, 2))

PyTorch version: 2.11.0+cu128
CUDA available: True


W0709 10:43:53.447000 546 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX export successful: resnet18_static.onnx
Input name: input
Output name: output
Number of nodes: 49
File size MB: 0.08
